In [1]:
import pandas as pd
import re
import os
import nltk
import html
import contractions
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.corpus import wordnet
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm




In [2]:
# Download required NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('averaged_perceptron_tagger')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\user\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[

True

In [3]:
TRAIN_PATH   = '../data/raw/train.csv'
TEST_PATH    = '../data/raw/test.csv'
OUTPUT_DIR   = '../data/processed'
SAMPLE_SIZE  = None

os.makedirs(OUTPUT_DIR, exist_ok=True)


## Load Data

In [4]:
df = pd.read_csv(
    TRAIN_PATH,
    header=0,
    names=['label', 'text'],
    nrows=SAMPLE_SIZE
)

print(f'Shape   : {df.shape}')
df.head()


Shape   : (650000, 2)


,label,text
0,4,dr. goldberg offers everything i look for in a...
1,1,"Unfortunately, the frustration of being Dr. Go..."
2,3,Been going to Dr. Goldberg for over 10 years. ...
3,3,Got a letter in the mail last week that said D...
4,0,I don't know what Dr. Goldberg was like before...


## Data Cleaning

In [5]:
# Missing values
df.isnull().sum()

label    0
text     0
dtype: int64

In [6]:
# Duplicate rows
df.duplicated().sum()

0

In [7]:
df.dropna(subset=['text'], inplace=True)
df.drop_duplicates(subset=['text'], inplace=True)
df.reset_index(drop=True, inplace=True)

In [8]:
#    Convert 5-class label to binary.
# Positive (1) = labels 3, 4  (4 & 5 stars)
# Negative (0) = labels 0, 1  (1 & 2 stars)
# Neutral  (2) = label  2     (3 stars)
def convert_to_binary_label(label: int) -> int:
    return 1 if label >= 3 else 0


## Preprocessing Pipeline

In [9]:
def expand_contractions(text: str) -> str:
    return contractions.fix(text)

In [10]:
# Lowercase
def to_lowercase(text: str) -> str:
    return text.lower()

In [11]:
# Remove URLs
def remove_urls(text: str) -> str:
    return re.sub(r'https?://\S+|www\.\S+', '', text)

In [12]:
# Remove HTML tags
def remove_html_tags(text: str) -> str:
    text = html.unescape(text)
    return re.sub(r'<.*?>', ' ', text)

In [ ]:
# removes digits & symbols
def remove_special_characters(text: str) -> str:
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'n(?=[a-z])', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [15]:
# Split text into individual word tokens
def tokenize(text: str) -> list:
    return word_tokenize(text)

In [16]:
# Remove stopwords but KEEP negation words
base_stopwords = set(stopwords.words('english'))

negations = {'no', 'not', 'nor', 'none', 'neither', 'never', 'very', 'too'}

STOP_WORDS = base_stopwords - negations

def remove_stopwords(tokens: list) -> list:
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]


In [17]:
lemmatizer = WordNetLemmatizer()

def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def lemmatize_with_pos(tokens: list) -> list:
    pos_tags = nltk.pos_tag(tokens)
    return [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in pos_tags if word.strip()
    ]

In [18]:
def preprocess_text(text: str) -> str:
    text = expand_contractions(text)
    text = remove_urls(text)
    text = remove_html_tags(text)
    text = to_lowercase(text)
    text = remove_special_characters(text)

    tokens = tokenize(text)
    tokens = remove_stopwords(tokens)
    tokens = lemmatize_with_pos(tokens)

    return ' '.join(tokens)

In [19]:
# Drop neutral reviews
original_len = len(df)
df = df[df['label'] != 2].reset_index(drop=True)
print(f'Removed neutral reviews : {original_len - len(df):,}')
print(f'Remaining rows : {len(df):,}')

Removed neutral reviews : 130,000
Remaining rows : 520,000


In [20]:
tqdm.pandas()
df['binary_label'] = df['label'].apply(convert_to_binary_label)
df['cleaned_text'] = df['text'].progress_apply(preprocess_text)


100%|██████████| 520000/520000 [35:49<00:00, 241.94it/s]  


In [21]:
# Word count statistics
df['original_word_count'] = df['text'].apply(lambda x: len(x.split()))
df['cleaned_word_count']  = df['cleaned_text'].apply(lambda x: len(x.split()))

print('WORD COUNT STATISTICS')
print('\nOriginal text:')
print(df['original_word_count'].describe().round(1))
print('\nCleaned text:')
print(df['cleaned_word_count'].describe().round(1))

reduction = (1 - df['cleaned_word_count'].mean() / df['original_word_count'].mean()) * 100
print(f'\nAverage word reduction: {reduction:.1f}%')

WORD COUNT STATISTICS

Original text:
count    520000.0
mean        133.0
std         122.5
min           1.0
25%          51.0
50%          97.0
75%         173.0
max        1052.0
Name: original_word_count, dtype: float64

Cleaned text:
count    520000.0
mean         67.8
std          61.3
min           0.0
25%          27.0
50%          50.0
75%          88.0
max         619.0
Name: cleaned_word_count, dtype: float64

Average word reduction: 49.0%


In [22]:
# Filtering short reviews
original_len = len(df)

df = df[df['cleaned_text'].apply(lambda x: len(str(x).split()) >= 2)].reset_index(drop=True)

print(f'Rows removed: {original_len - len(df)}')
print(f'Remaining rows: {len(df):,}')

Rows removed: 492
Remaining rows: 519,508


In [23]:
# Before vs After preview
for i, row in df.sample(3, random_state=42).iterrows():
    print(f"\n[binary_label: {row['binary_label']}]")
    print(f"  Original : {row['text'][:200]}")
    print(f"  Cleaned  : {row['cleaned_text'][:200]}")


[binary_label: 0]
  Original : I usually do not rate places when I travel. But this place deserves a bad rating. Was in town for business.\n\nFood- not good. \nService- eating dinner instead of serving me.\n\nPretty bad overall.
  Cleaned  : usually not rate place travel place deserves bad rating town business nfood not good nservice eat dinner instead serve npretty bad overall

[binary_label: 1]
  Original : In a nutshell:  a unique bar, but very, very pricey.\nPros:  This place is stylish, and the concept of having shadow dancers behind the screen was unique, but has been imitated.  \nCons: You can see t
  Cleaned  : nutshell unique bar very very pricey npros place stylish concept shadow dancer behind screen unique imitated ncons see stage casino outside not need buy drink overpriced appetizer hang

[binary_label: 1]
  Original : You know that mythical hole-in-the-wall sushi place, the one run by a local proprietor and serves delicious food at reasonable prices that no one ever know

## Save Train Set

In [24]:
df_train_final = df[['binary_label', 'text', 'cleaned_text']].copy()

print(f'Shape   : {df_train_final.shape}')
df_train_final.head()

Shape   : (519508, 3)


,binary_label,text,cleaned_text
0,1,dr. goldberg offers everything i look for in a...,goldberg offer everything look general practit...
1,0,"Unfortunately, the frustration of being Dr. Go...",unfortunately frustration goldberg patient rep...
2,1,Been going to Dr. Goldberg for over 10 years. ...,go goldberg year think one patient start mhmg ...
3,1,Got a letter in the mail last week that said D...,get letter mail last week say goldberg move ar...
4,0,I don't know what Dr. Goldberg was like before...,not know goldberg like move arizona let tell s...


In [25]:
train_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_train.csv')
df_train_final.to_csv(train_output_path, index=False)


In [26]:
# Train label distribution
print('Label distribution (Train set):')
dist = df_train_final['binary_label'].value_counts().sort_index()
total = len(df_train_final)

for label, count in dist.items():
    sentiment = 'Positive' if label == 1 else 'Negative'
    print(f'  {sentiment} ({label}) : {count:>7,} rows ({count/total*100:5.1f}%)')

print(f'\n  Total : {total:>7,} rows')

Label distribution (Train set):
  Negative (0) : 259,691 rows ( 50.0%)
  Positive (1) : 259,817 rows ( 50.0%)

  Total : 519,508 rows


## Apply Pipeline — Test Set

In [27]:
df_test = pd.read_csv(TEST_PATH, header=0, names=['label', 'text'])

df_test.dropna(subset=['text'], inplace=True)
df_test.drop_duplicates(subset=['text'], inplace=True)
df_test.reset_index(drop=True, inplace=True)

df_test = df_test[df_test['label'] != 2].reset_index(drop=True)

df_test['binary_label'] = df_test['label'].apply(convert_to_binary_label)
df_test['cleaned_text'] = df_test['text'].progress_apply(preprocess_text)

# Drop rows where cleaned_text has less than 2 words
df_test = df_test[df_test['cleaned_text'].apply(lambda x: len(str(x).split()) >= 2)].reset_index(drop=True)


df_test_final = df_test[['binary_label', 'text', 'cleaned_text']].copy()

print(f'Test set ready : {len(df_test_final):,} rows')
df_test_final.head()

100%|██████████| 40000/40000 [01:49<00:00, 364.91it/s]


Test set ready : 39,966 rows


,binary_label,text,cleaned_text
0,0,I got 'new' tires from them and within two wee...,get new tire within two week get flat take car...
1,0,Don't waste your time. We had two different p...,not waste time two different people come house...
2,0,All I can say is the worst! We were the only 2...,say bad people place lunch place freeze loaded...
3,0,I have been to this restaurant twice and was d...,restaurant twice disappoint time not back firs...
4,0,Food was NOT GOOD at all! My husband & I ate h...,food not good husband ate couple week ago firs...


## Save Test Set

In [28]:
test_output_path = os.path.join(OUTPUT_DIR, 'preprocessed_test.csv')
df_test_final.to_csv(test_output_path, index=False)

In [29]:
# Test label distribution
print('Label distribution (Test set):')
dist = df_test_final['binary_label'].value_counts().sort_index()
total = len(df_test_final)

for label, count in dist.items():
    sentiment = 'Positive' if label == 1 else 'Negative'
    print(f'  {sentiment} ({label}) : {count:>7,} rows ({count/total*100:5.1f}%)')

print(f'\n  Total : {total:>7,} rows')

Label distribution (Test set):
  Negative (0) :  19,977 rows ( 50.0%)
  Positive (1) :  19,989 rows ( 50.0%)

  Total :  39,966 rows
